In [0]:
# Display all tables in silver schema
print("="*60)
print("SILVER LAYER TRANSFORMATION SUMMARY")
print("="*60)

tables = spark.sql("SHOW TABLES IN my_catalog.silver").collect()

print(f"\nTotal tables created: {len(tables)}\n")

# Get record counts for each table
from pyspark.sql.types import StructType, StructField, StringType, LongType

schema = StructType([
    StructField("Table_Name", StringType(), True),
    StructField("Record_Count", LongType(), True)
])

table_stats = []

for table in tables:
    table_name = table['tableName']
    try:
        count = spark.table(f"my_catalog.silver.{table_name}").count()
        table_stats.append((table_name, count))
        print(f"✓ {table_name:30s} : {count:,} records")
    except Exception as e:
        print(f"✗ {table_name:30s} : Error - {str(e)}")

print("\n" + "="*60)
print("TRANSFORMATION COMPLETE")
print("="*60)

# Create a summary DataFrame
df_summary = spark.createDataFrame(table_stats, schema)
display(df_summary)

In [0]:
# Read from bronze layer
df_staff_locations = spark.table("my_catalog.bronze.staff_locations_view")

# Get schema and apply basic transformations
cols = df_staff_locations.columns

# Apply standardization to string columns
df_staff_locations_transformed = df_staff_locations
for col in cols:
    col_type = dict(df_staff_locations.dtypes)[col]
    if col_type == 'string':
        df_staff_locations_transformed = df_staff_locations_transformed.withColumn(col, F.trim(F.col(col)))

# Add audit columns
df_staff_locations_final = df_staff_locations_transformed \
    .withColumn("Load_Timestamp", F.current_timestamp()) \
    .withColumn("Processing_Date", F.current_date())

# Data Quality Check
record_count = df_staff_locations_final.count()

# Write to silver layer
df_staff_locations_final.write.mode("overwrite").saveAsTable("my_catalog.silver.staff_locations_view")

print(f"✓ Staff Locations View table transformed and loaded to silver layer")
print(f"  - Total records: {record_count}")

In [0]:
# Read from bronze layer
df_product_info = spark.table("my_catalog.bronze.product_info_m_view")

# Get schema and apply basic transformations
cols = df_product_info.columns

# Apply standardization to string columns
df_product_info_transformed = df_product_info
for col in cols:
    col_type = dict(df_product_info.dtypes)[col]
    if col_type == 'string':
        df_product_info_transformed = df_product_info_transformed.withColumn(col, F.trim(F.col(col)))

# Add audit columns
df_product_info_final = df_product_info_transformed \
    .withColumn("Load_Timestamp", F.current_timestamp()) \
    .withColumn("Processing_Date", F.current_date())

# Data Quality Check
record_count = df_product_info_final.count()

# Write to silver layer
df_product_info_final.write.mode("overwrite").saveAsTable("my_catalog.silver.product_info_m_view")

print(f"✓ Product Info View table transformed and loaded to silver layer")
print(f"  - Total records: {record_count}")

In [0]:
# Read from bronze layer
df_organizations = spark.table("my_catalog.bronze.organizations")

# Data Quality: Remove duplicates based on Organization_Id
window_spec = Window.partitionBy("Organization_Id").orderBy(F.col("Index").desc())
df_organizations_dedup = df_organizations.withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")

# Standardization
df_organizations_transformed = df_organizations_dedup \
    .withColumn("Name", F.trim(F.col("Name"))) \
    .withColumn("Country", F.upper(F.trim(F.col("Country")))) \
    .withColumn("Industry", F.initcap(F.trim(F.col("Industry")))) \
    .withColumn("Description", F.trim(F.col("Description"))) \
    .withColumn("Website", F.lower(F.trim(F.col("Website"))))

# Business Logic: Add derived columns
df_organizations_final = df_organizations_transformed \
    .withColumn("Company_Age", F.year(F.current_date()) - F.col("Founded")) \
    .withColumn("Company_Age_Group", 
        F.when(F.col("Company_Age") < 5, "Startup")
         .when(F.col("Company_Age") < 10, "Growing")
         .when(F.col("Company_Age") < 25, "Established")
         .otherwise("Legacy")) \
    .withColumn("Company_Size", 
        F.when(F.col("Number_of_employees") < 50, "Small")
         .when(F.col("Number_of_employees") < 250, "Medium")
         .when(F.col("Number_of_employees") < 1000, "Large")
         .otherwise("Enterprise")) \
    .withColumn("Has_Website", F.col("Website").isNotNull()) \
    .withColumn("Is_Recent_Founded", 
        F.when(F.col("Founded") >= F.year(F.current_date()) - 5, True)
         .otherwise(False)) \
    .withColumn("Load_Timestamp", F.current_timestamp()) \
    .withColumn("Processing_Date", F.current_date())

# Data Quality Checks
record_count = df_organizations_final.count()
avg_employees = df_organizations_final.agg(F.avg("Number_of_employees")).collect()[0][0]
avg_age = df_organizations_final.agg(F.avg("Company_Age")).collect()[0][0]
with_website = df_organizations_final.filter(F.col("Has_Website") == True).count()

# Write to silver layer
df_organizations_final.write.mode("overwrite").saveAsTable("my_catalog.silver.organizations")

print(f"✓ Organizations table transformed and loaded to silver layer")
print(f"  - Total records: {record_count}")
print(f"  - Average employees: {avg_employees:.0f}")
print(f"  - Average company age: {avg_age:.1f} years")
print(f"  - With website: {with_website} ({with_website/record_count*100:.2f}%)")

In [0]:
# Read from bronze layer
df_people = spark.table("my_catalog.bronze.people")

# Data Quality: Remove duplicates based on User_Id
window_spec = Window.partitionBy("User_Id").orderBy(F.col("Index").desc())
df_people_dedup = df_people.withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")

# Standardization
df_people_transformed = df_people_dedup \
    .withColumn("Email", F.lower(F.trim(F.col("Email")))) \
    .withColumn("First_Name", F.initcap(F.trim(F.col("First_Name")))) \
    .withColumn("Last_Name", F.initcap(F.trim(F.col("Last_Name")))) \
    .withColumn("Full_Name", F.concat_ws(" ", F.col("First_Name"), F.col("Last_Name"))) \
    .withColumn("Phone", F.regexp_replace(F.col("Phone"), "[^0-9+]", "")) \
    .withColumn("Sex", F.upper(F.trim(F.col("Sex")))) \
    .withColumn("Job_Title", F.trim(F.col("Job_Title")))

# Business Logic: Calculate age and age groups
df_people_final = df_people_transformed \
    .withColumn("Age", F.floor(F.months_between(F.current_date(), F.col("Date_of_birth")) / 12)) \
    .withColumn("Age_Group", 
        F.when(F.col("Age") < 18, "Minor")
         .when(F.col("Age") < 30, "18-29")
         .when(F.col("Age") < 40, "30-39")
         .when(F.col("Age") < 50, "40-49")
         .when(F.col("Age") < 60, "50-59")
         .otherwise("60+")) \
    .withColumn("Is_Working_Age", 
        F.when((F.col("Age") >= 18) & (F.col("Age") <= 65), True)
         .otherwise(False)) \
    .withColumn("Gender", 
        F.when(F.col("Sex") == "M", "Male")
         .when(F.col("Sex") == "F", "Female")
         .otherwise("Other")) \
    .withColumn("Load_Timestamp", F.current_timestamp()) \
    .withColumn("Processing_Date", F.current_date())

# Data Quality Checks
record_count = df_people_final.count()
valid_email_count = df_people_final.filter(F.col("Email").rlike("^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$")).count()
avg_age = df_people_final.agg(F.avg("Age")).collect()[0][0]

# Write to silver layer
df_people_final.write.mode("overwrite").saveAsTable("my_catalog.silver.people")

print(f"✓ People table transformed and loaded to silver layer")
print(f"  - Total records: {record_count}")
print(f"  - Valid emails: {valid_email_count} ({valid_email_count/record_count*100:.2f}%)")
print(f"  - Average age: {avg_age:.1f} years")

In [0]:
# Read from bronze layer
df_products = spark.table("my_catalog.bronze.products")

# Data Quality: Remove duplicates based on Internal_ID
window_spec = Window.partitionBy("Internal_ID").orderBy(F.col("Index").desc())
df_products_dedup = df_products.withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")

# Standardization
df_products_transformed = df_products_dedup \
    .withColumn("Name", F.trim(F.col("Name"))) \
    .withColumn("Brand", F.initcap(F.trim(F.col("Brand")))) \
    .withColumn("Category", F.initcap(F.trim(F.col("Category")))) \
    .withColumn("Description", F.trim(F.col("Description"))) \
    .withColumn("Currency", F.upper(F.trim(F.col("Currency")))) \
    .withColumn("EAN", F.col("EAN").cast("string"))

# Business Logic: Add derived columns
df_products_final = df_products_transformed \
    .withColumn("Stock_Status", 
        F.when(F.col("Stock") == 0, "Out of Stock")
         .when(F.col("Stock") < 10, "Low Stock")
         .when(F.col("Stock") < 50, "Medium Stock")
         .otherwise("In Stock")) \
    .withColumn("Price_Range", 
        F.when(F.col("Price") < 50, "Budget")
         .when(F.col("Price") < 200, "Mid-Range")
         .when(F.col("Price") < 500, "Premium")
         .otherwise("Luxury")) \
    .withColumn("Is_Available", 
        F.when(F.col("Availability") == "Yes", True)
         .otherwise(False)) \
    .withColumn("Has_Size_Options", F.col("Size").isNotNull()) \
    .withColumn("Has_Color_Options", F.col("Color").isNotNull()) \
    .withColumn("Load_Timestamp", F.current_timestamp()) \
    .withColumn("Processing_Date", F.current_date())

# Data Quality Checks
record_count = df_products_final.count()
in_stock_count = df_products_final.filter(F.col("Stock") > 0).count()
available_count = df_products_final.filter(F.col("Is_Available") == True).count()
avg_price = df_products_final.agg(F.avg("Price")).collect()[0][0]

# Write to silver layer
df_products_final.write.mode("overwrite").saveAsTable("my_catalog.silver.products")

print(f"✓ Products table transformed and loaded to silver layer")
print(f"  - Total records: {record_count}")
print(f"  - In stock: {in_stock_count} ({in_stock_count/record_count*100:.2f}%)")
print(f"  - Available: {available_count} ({available_count/record_count*100:.2f}%)")
print(f"  - Average price: {avg_price:.2f}")

In [0]:
# Read from bronze layer
df_leads = spark.table("my_catalog.bronze.leads")

# Data Quality: Remove duplicates based on Account_Id
window_spec = Window.partitionBy("Account_Id").orderBy(F.col("Index").desc())
df_leads_dedup = df_leads.withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")

# Standardization
df_leads_transformed = df_leads_dedup \
    .withColumn("Email_1", F.lower(F.trim(F.col("Email_1")))) \
    .withColumn("Email_2", F.lower(F.trim(F.col("Email_2")))) \
    .withColumn("First_Name", F.initcap(F.trim(F.col("First_Name")))) \
    .withColumn("Last_Name", F.initcap(F.trim(F.col("Last_Name")))) \
    .withColumn("Full_Name", F.concat_ws(" ", F.col("First_Name"), F.col("Last_Name"))) \
    .withColumn("Phone_1", F.regexp_replace(F.col("Phone_1"), "[^0-9+]", "")) \
    .withColumn("Phone_2", F.regexp_replace(F.col("Phone_2"), "[^0-9+]", "")) \
    .withColumn("Lead_Owner", F.initcap(F.trim(F.col("Lead_Owner"))))

# Business Logic: Categorize deal stages
df_leads_final = df_leads_transformed \
    .withColumn("Deal_Stage_Category", 
        F.when(F.col("Deal_Stage").isin(["Closed Won", "Won"]), "Converted")
         .when(F.col("Deal_Stage").isin(["Closed Lost", "Lost"]), "Lost")
         .when(F.col("Deal_Stage").isin(["Negotiation", "Proposal", "Qualification"]), "Active")
         .otherwise("New")) \
    .withColumn("Primary_Email", F.coalesce(F.col("Email_1"), F.col("Email_2"))) \
    .withColumn("Has_Multiple_Contacts", 
        F.when((F.col("Email_1").isNotNull()) & (F.col("Email_2").isNotNull()), True)
         .otherwise(False)) \
    .withColumn("Load_Timestamp", F.current_timestamp()) \
    .withColumn("Processing_Date", F.current_date())

# Data Quality Checks
record_count = df_leads_final.count()
active_leads = df_leads_final.filter(F.col("Deal_Stage_Category") == "Active").count()
converted_leads = df_leads_final.filter(F.col("Deal_Stage_Category") == "Converted").count()

# Write to silver layer
df_leads_final.write.mode("overwrite").saveAsTable("my_catalog.silver.leads")

print(f"✓ Leads table transformed and loaded to silver layer")
print(f"  - Total records: {record_count}")
print(f"  - Active leads: {active_leads} ({active_leads/record_count*100:.2f}%)")
print(f"  - Converted leads: {converted_leads} ({converted_leads/record_count*100:.2f}%)")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

# Create silver schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS my_catalog.silver")

print("✓ Libraries imported and silver schema created")

In [0]:
# Read from bronze layer
df_customers = spark.table("my_catalog.bronze.customers")

# Data Quality: Remove duplicates based on Customer_Id
window_spec = Window.partitionBy("Customer_Id").orderBy(F.col("Index").desc())
df_customers_dedup = df_customers.withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")

# Standardization
df_customers_transformed = df_customers_dedup \
    .withColumn("Email", F.lower(F.trim(F.col("Email")))) \
    .withColumn("First_Name", F.initcap(F.trim(F.col("First_Name")))) \
    .withColumn("Last_Name", F.initcap(F.trim(F.col("Last_Name")))) \
    .withColumn("Full_Name", F.concat_ws(" ", F.col("First_Name"), F.col("Last_Name"))) \
    .withColumn("Phone_1", F.regexp_replace(F.col("Phone_1"), "[^0-9+]", "")) \
    .withColumn("Phone_2", F.regexp_replace(F.col("Phone_2"), "[^0-9+]", "")) \
    .withColumn("Country", F.upper(F.trim(F.col("Country")))) \
    .withColumn("City", F.initcap(F.trim(F.col("City"))))

# Business Logic: Calculate customer tenure
df_customers_final = df_customers_transformed \
    .withColumn("Customer_Tenure_Days", F.datediff(F.current_date(), F.col("Subscription_Date"))) \
    .withColumn("Is_Active_Customer", F.when(F.col("Email").isNotNull(), True).otherwise(False)) \
    .withColumn("Load_Timestamp", F.current_timestamp()) \
    .withColumn("Processing_Date", F.current_date())

# Data Quality Checks
record_count = df_customers_final.count()
valid_email_count = df_customers_final.filter(F.col("Email").rlike("^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$")).count()

# Write to silver layer
df_customers_final.write.mode("overwrite").saveAsTable("my_catalog.silver.customers")

print(f"✓ Customers table transformed and loaded to silver layer")
print(f"  - Total records: {record_count}")
print(f"  - Valid emails: {valid_email_count} ({valid_email_count/record_count*100:.2f}%)")